# Supplementary Figure 18 | Joint ColocBoost recovers an OLIG-locus eosinophil GWAS colocalization

**a.** The -log10 (p-value) for each tested variant is plotted against its physical position on chromosome 21. The lead CoS variant is represented by a purple diamond (chr21:33069392:T:A), other variants are color-coded by their linkage disequilibrium (r²) with the lead SNP according to GTEx European LD (red: r² ≥ 0.8; orange: 0.6 ≤ r² < 0.8; green: 0.4 ≤ r² < 0.6; light blue: 0.2 ≤ r² < 0.4; dark blue: r² < 0.2). Bottom tracks show ColocBoost variant colocalization probability (VCP).

**b.** Gene annotations and transcript structures from GENCODE v47 in the locus window and zoomed in around the CoS lead.

**c.** Median expression (log10(TPM + 1)) of the four CoS genes in each GTEx tissue, sorted by novel lncRNA expression. Black box marks whole blood, the tissue of this colocalization.

**d.** LocusCompare plots of eosinophil count (EC) GWAS -log10 (p-value) versus each CoS gene’s whole blood eQTL -log10 (p-value). Inset gives CoS normalized probability of colocalization (CoSnpc) for each gene (where applicable) in a joint ColocBoost model with all genes, all genes except novel lncRNA, a protein-coding only model, and a single-phenotype model considering each eQTL one at a time. Only the joint model colocalizes (CoSnpc > 0.75).

**e.** CoSnpc and NPCoutcome for single-phenotype, protein-coding-only joint, no-novel-lncRNA joint, and all-gene joint models. Although NPCoutcome remains high across models, indicating that ColocBoost is still mapping effects on individual phenotypes, the GWAS locus joins a multi-gene CoS with CoSnpc > 0.75 and NPCoutcome > 0.2 only in the all-gene joint model.


## Setup


In [ ]:
import glob
import gzip
import json
import lzma
import os
import re
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam
import seaborn as sns
from matplotlib.collections import LineCollection
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Rectangle
from matplotlib.transforms import blended_transform_factory
from tqdm.auto import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.6, 5.1, 300

tissue = "Whole_Blood"
region_id = "chr21_30836162_33278486"
cos_lead_variant = "chr21:33069392:T:A"
gwas_primary = "Astle_et_al_2016_Eosinophil_counts"
gene_ids = [
    "ENSG00000184221.13",  # OLIG1
    "ENSG00000205927.5",   # OLIG2
    "ENSG00000227757.4",   # existing lncRNA
    "ENSG00000301393.1",   # novel lncRNA
]
cos_npc_thresh = 0.75
npc_outcome_min = 0.2
locus_win_half = 100_000
zoom_win_half = 25_000

ld_bounds = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
ld_colors = ["black", "#4298B5", "#6FA287", "#E98300", "#B83A4B"]
ld_cmap = ListedColormap(ld_colors)
ld_norm = BoundaryNorm(ld_bounds, ld_cmap.N)
base_marker_s = 4
gene_type_palette = {
    "protein_coding": sns.color_palette("Paired", 12)[1],
    "lncRNA": sns.color_palette("Paired", 12)[5],
    "novel_lncRNA": sns.color_palette("Paired", 12)[5],
    "existing_lncRNA": sns.color_palette("Paired", 12)[4],
    "gwas": "#222222",
}
pc_blue_shades = [
    sns.color_palette("Paired", 12)[0],
    sns.color_palette("Paired", 12)[1],
]

# Zenodo ColocBoost helpers
def tissue_from_cos_id(cid):
    s = str(cid)
    i = s.rfind("_")
    return s[:i] if i > 0 else s


def lead_from_lists(variants, vcp):
    vs = list(variants)
    vcps = list(vcp)
    ps = pd.to_numeric(pd.Series(vcps), errors="coerce")
    if len(ps) == len(vs) and ps.notna().any():
        return vs[int(ps.values.argmax())]
    return vs[0]


def cos_to_long(row):
    vs = list(row["variants"])
    ps = list(row["vcp"])
    return pd.DataFrame({
        "variant_id": vs,
        "vcp": ps[: len(vs)] + [float("nan")] * (len(vs) - len(ps)),
    })


def _coloc_list_len(val):
    if val is None:
        return 0
    if isinstance(val, float) and np.isnan(val):
        return 0
    if isinstance(val, np.ndarray):
        return len(val)
    if isinstance(val, (list, tuple, set)):
        return len(val)
    return 0


def _typeset(vals):
    if vals is None:
        return set()
    if isinstance(vals, float) and np.isnan(vals):
        return set()
    if isinstance(vals, np.ndarray):
        items = vals.tolist()
    elif isinstance(vals, (list, tuple, set)):
        items = list(vals)
    else:
        items = [vals]
    out = set()
    for v in items:
        if v is None:
            continue
        s = str(v).strip()
        if s and s not in ("nan", "None", "<NA>"):
            out.add(s)
    return out


def _member_qtl_class(types):
    t = _typeset(types)
    lnc = t & {"novel_lncRNA", "existing_lncRNA"}
    non = t - {"novel_lncRNA", "existing_lncRNA"}
    if not lnc:
        return "non-lncRNA"
    if non:
        return "Shared"
    return "lncRNA-specific (robust)"


def _locus_qtl_label(member_classes):
    labs = set(member_classes)
    if labs == {"lncRNA-specific (robust)"}:
        return "lncRNA-specific (robust)"
    if "Shared" in labs or ("non-lncRNA" in labs and "lncRNA-specific (robust)" in labs):
        return "Shared"
    return "non-lncRNA"


def load_main_colocboost(colocboost_dir, *, cols=None):
    frames = []
    for path in sorted(colocboost_dir.glob("*.ColocBoost.parquet")):
        if path.name.startswith("cross_"):
            continue
        tissue = path.name.replace(".ColocBoost.parquet", "")
        df = pd.read_parquet(path, columns=cols)
        df["tissue"] = tissue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_cross_tissue_members(cross_tissue_loci_path, colocboost_dir, *, include_ucos=False):
    loci = pd.read_parquet(cross_tissue_loci_path)
    all_cos = load_main_colocboost(colocboost_dir).set_index("cos_id", drop=False)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    cos_rows = memb[["locus_id", "cos_id"]].merge(
        all_cos.reset_index(drop=True), on="cos_id", how="inner"
    )
    cos_rows["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(cos_rows["variants"], cos_rows["vcp"])
    ]
    if "phenotype_id" not in cos_rows.columns:
        cos_rows["phenotype_id"] = cos_rows["phenotype_ids"]

    qtl_by_locus = (
        cos_rows.groupby("locus_id")["gene_types_robust"]
        .agg(lambda s: _locus_qtl_label(_member_qtl_class(x) for x in s))
    )
    locus_summary = loci.copy()
    locus_summary["qtl_label"] = locus_summary["locus_id"].map(qtl_by_locus)

    if not include_ucos:
        return locus_summary, cos_rows

    member_ids = set(memb["cos_id"])
    has_gwas = all_cos["gwas_traits"].map(_coloc_list_len) > 0
    has_qtl = all_cos["gene_ids_robust"].map(_coloc_list_len) > 0
    ucos = all_cos.loc[
        ~all_cos.index.isin(member_ids) & (has_gwas ^ has_qtl)
    ].reset_index(drop=True)
    ucos = ucos.loc[has_gwas.reindex(ucos["cos_id"]).fillna(False).values].copy()
    ucos["locus_id"] = "ucos_" + ucos["cos_id"].astype(str)
    ucos["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(ucos["variants"], ucos["vcp"])
    ]
    if "phenotype_id" not in ucos.columns:
        ucos["phenotype_id"] = ucos["phenotype_ids"]
    members = pd.concat([cos_rows, ucos], ignore_index=True)
    return locus_summary, members




## Paths

In [ ]:
annot_feather = Path("path/to/gencode.v47.annotation.gtf.feather")
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
eqtl_nominal_dir = Path("path/to/v11_eqtl_allpairs")
gwas_nominal_dir = Path("path/to/imputed_gwas")
ld_dir = Path("path/to/ld_blocks_gtex_eur")
trait_abbr_tsv = Path("path/to/table_s2_abbreviations.txt")
gwas_meta_file = Path("path/to/barbeira_full_gwas_metadata.txt")
tpm_medians_file = Path("path/to/tissue_medians.txt")
gtex_colors_json = Path("refs/colors.json")

joint_path = Path(f"path/to/{tissue}.ColocBoost.parquet")
pc_path = Path(f"path/to/{tissue}.only_protein_coding.ColocBoost.parquet")
v39_path = Path(f"path/to/{tissue}.only_v39.ColocBoost.parquet")
sg_path = Path("path/to/Whole_Blood.single_phenotype.ColocBoost.parquet")

fig_dir = Path("figures")


## Load data


### Reference annotations and expression


In [ ]:
# GWAS short labels
s2_abbr = set()
with open(trait_abbr_tsv) as fh:
    fh.readline()
    for line in fh:
        parts = line.strip().split()
        if parts:
            s2_abbr.add(parts[-2])
gwas_meta = pd.read_csv(gwas_meta_file, sep="\t")
trait_labels = {
    str(tag): str(code)
    for tag, code in zip(gwas_meta["Tag"], gwas_meta["new_abbreviation"])
    if pd.notna(code) and code in s2_abbr
}

# Gene annotation + lncRNA split classes
ann = pd.read_feather(annot_feather)
gene_ann = ann[ann["Feature"] == "gene"][
    ["gene_id", "gene_name", "gene_type", "Chromosome", "Start", "End", "Strand"]
].drop_duplicates("gene_id")
gene_ann["stable"] = gene_ann["gene_id"].str.split(".").str[0]
stable_to_name = dict(zip(gene_ann["stable"], gene_ann["gene_name"]))
stable_to_type = dict(zip(gene_ann["stable"], gene_ann["gene_type"]))
gclass = pd.read_table(gencode_class, low_memory=False)
split_map = dict(zip(gclass["gene_id_stable"].astype(str), gclass["gene_type_grouped"]))


def gene_display_name(gid):
    return stable_to_name.get(str(gid).split(".")[0], str(gid).split(".")[0])


def gene_biotype_key(gid):
    core = str(gid).split(".")[0]
    split = split_map.get(core)
    if isinstance(split, str) and split and split != "protein_coding":
        return split
    return "protein_coding" if stable_to_type.get(core) == "protein_coding" else "lncRNA"


def gwas_short_label(gwas_id):
    return trait_labels.get(str(gwas_id), str(gwas_id).replace("_", " ")[:40])


# GTEx colors + median TPM expression matrix for CoS genes
with open(gtex_colors_json) as fh:
    gtex_color_meta = json.load(fh)
tissue_color = {
    t: f"#{str(m.get('tissue_color_hex', '888888')).lstrip('#')}"
    for t, m in gtex_color_meta.items()
}
tissue_abbr = {t: str(m.get("tissue_abbrv", t[:6])) for t, m in gtex_color_meta.items()}

median_tpm = pd.read_table(tpm_medians_file, index_col=0)
expr_tissues = [t for t in gtex_color_meta if t in median_tpm.columns and t.count(" - ") <= 1]
expr_mat = median_tpm.reindex(gene_ids)[expr_tissues]
novel_order = (
    expr_mat.loc["ENSG00000301393.1"].astype(float).sort_values(ascending=False).index.tolist()
)
expr_log = np.log10(expr_mat[novel_order].astype(float) + 1.0)

print("Trait labels:", len(trait_labels))
print("Genes:", {gid: gene_display_name(gid) for gid in gene_ids})
print(f"Expression: {expr_log.shape[0]} genes × {expr_log.shape[1]} tissues")

### ColocBoost model outputs (Zenodo tissue leads / long)


In [ ]:
def _pheno_col(df):
    return "phenotype_ids" if "phenotype_ids" in df.columns else "phenotype_id"


def _prep_colocboost_table(path):
    df = pd.read_parquet(path)
    ph = _pheno_col(df)
    if "phenotype_id" not in df.columns:
        df = df.copy()
        df["phenotype_id"] = df[ph]
    df["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(df["variants"], df["vcp"])
    ]
    hit = df["variants"].map(lambda vs: cos_lead_variant in list(vs))
    return df.loc[hit].reset_index(drop=True)


full_df_leads = _prep_colocboost_table(joint_path)
pc_df = _prep_colocboost_table(pc_path)
v39_df = _prep_colocboost_table(v39_path)
sg_df = _prep_colocboost_table(sg_path)

_joint = pd.read_parquet(joint_path)
_ph = _pheno_col(_joint)
_focal = _joint[
    _joint["variants"].map(lambda vs: cos_lead_variant in list(vs))
].copy()
if _focal.empty:
    raise ValueError(f"No CoS with lead {cos_lead_variant} in {joint_path}")
# Prefer the joint CoS that also includes the example GWAS trait.
_gwas_hit = _focal[_ph].astype(str).str.contains(gwas_primary, regex=False)
if _gwas_hit.any():
    _focal = _focal.loc[_gwas_hit]
_focal["_max_vcp"] = [
    float(pd.to_numeric(pd.Series(list(v)), errors="coerce").max())
    for v in _focal["vcp"]
]
focal_row = _focal.sort_values("_max_vcp", ascending=False).iloc[0]
focal_cos_id = str(focal_row["cos_id"])

cos_long = cos_to_long(focal_row)
if cos_long.empty:
    raise ValueError(f"No variant/VCP rows for focal CoS {focal_cos_id}")
full_df = cos_long.copy()
full_df["cos_id"] = focal_cos_id
full_df["phenotype_id"] = focal_row[_ph]
full_df["cos_npc"] = focal_row["cos_npc"]
if "npc_outcomes" in focal_row.index:
    full_df["npc_outcomes"] = focal_row["npc_outcomes"]

lead_pos = int(cos_lead_variant.split(":")[1])
lead_panel_id = cos_lead_variant.replace(":", "_") + "_b38"
print(
    f"Loaded Zenodo ColocBoost: joint={len(full_df_leads)}, pc={len(pc_df)}, "
    f"v39={len(v39_df)}, single_phenotype={len(sg_df)}; "
    f"focal={focal_cos_id}; VCP variants={len(full_df):,}"
)


## Run analysis


### Build locus association table


In [ ]:
def parse_variant_full(vid):
    parts = str(vid).split(":")
    return parts[0], int(parts[1]), parts[2], parts[3]


def coloc_to_panel(vid):
    chrom, pos, ref, alt = parse_variant_full(vid)
    return f"{chrom}_{pos}_{ref}_{alt}_b38"


_ld_block_index = {}


def _variant_to_ld_id(variant_id):
    return str(variant_id).replace(":", "_")


def _resolve_ld_matrix_id(variant_id, ld_ids):
    ld_set = set(ld_ids)
    if ":" in str(variant_id):
        base = _variant_to_ld_id(variant_id)
        candidates = (base, f"{base}_b38", coloc_to_panel(variant_id))
    else:
        candidates = (str(variant_id),)
    for cand in candidates:
        if cand in ld_set:
            return cand
    if ":" in str(variant_id):
        chrom, pos, _, _ = parse_variant_full(variant_id)
        prefix = f"{chrom}_{pos}_"
        hits = sorted(v for v in ld_ids if v.startswith(prefix))
        if hits:
            return hits[0]
    return None


def _build_ld_block_index(chrom):
    chrom = chrom if str(chrom).startswith("chr") else f"chr{chrom}"
    rows = []
    for bim_path in glob.glob(str(ld_dir / f"LD_{chrom}_*.bim")):
        block_id = os.path.basename(bim_path).replace(".bim", "").replace("LD_", "")
        bim = pd.read_table(bim_path, header=None, usecols=[3], names=["pos"])
        rows.append((int(bim["pos"].min()), int(bim["pos"].max()), block_id))
    _ld_block_index[chrom] = sorted(rows)


def _ld_block_id_for_position(chrom, pos):
    chrom = chrom if str(chrom).startswith("chr") else f"chr{chrom}"
    if chrom not in _ld_block_index:
        _build_ld_block_index(chrom)
    for start, end, block_id in _ld_block_index[chrom]:
        if start <= pos <= end:
            return block_id
    raise ValueError(f"No LD block found for {chrom}:{pos}")


def _load_ld_r2_to_ref(ld_block_id, ld_ref_id, win_lo, win_hi):
    bim = pd.read_table(
        str(ld_dir / f"LD_{ld_block_id}.bim"), header=None,
        names=["chr", "variant_id", "cm", "pos", "a1", "a2"],
    )
    ref_hits = bim.index[bim["variant_id"] == ld_ref_id]
    if not len(ref_hits):
        return pd.Series(dtype=float)
    ref_row = int(ref_hits[0])
    n = len(bim)
    with lzma.open(str(ld_dir / f"LD_{ld_block_id}.ld.gz"), "rt") as fh:
        for i, line in enumerate(fh):
            if i == ref_row:
                vals = np.fromstring(line.strip(), sep="\t")
                break
        else:
            return pd.Series(dtype=float)
    if len(vals) > n:
        vals = vals[:n]
    elif len(vals) < n:
        vals = np.pad(vals, (0, n - len(vals)), constant_values=np.nan)
    ld_r2 = pd.Series(np.square(np.abs(vals)), index=bim["variant_id"].values)
    return ld_r2.loc[bim["pos"].between(win_lo, win_hi).values]


def load_ld_r2_for_locus(chrom, lead_variant, win_lo, win_hi):
    lead_pos = parse_variant_full(lead_variant)[1]
    block_id = _ld_block_id_for_position(chrom, lead_pos)
    bim = pd.read_table(
        str(ld_dir / f"LD_{block_id}.bim"), header=None,
        names=["chr", "variant_id", "cm", "pos", "a1", "a2"],
    )
    lead_ld_id = _resolve_ld_matrix_id(lead_variant, bim["variant_id"].tolist())
    if lead_ld_id is None:
        raise ValueError(f"Lead variant not found in LD block {block_id}")
    return _load_ld_r2_to_ref(block_id, lead_ld_id, win_lo, win_hi), coloc_to_panel(lead_variant)


def _tabix_chrom(chrom):
    c = str(chrom).strip()
    return c[3:] if c.lower().startswith("chr") else c


def load_gwas_nominal(gwas_id, chrom, win_lo, win_hi):
    path = str(gwas_nominal_dir / f"imputed_{gwas_id}.txt.gz")
    win_lo, win_hi = int(win_lo), int(win_hi)
    region = f"{_tabix_chrom(chrom)}:{win_lo}-{win_hi}"
    print(f"  GWAS tabix fetch: {gwas_id}  {region}", flush=True)
    with gzip.open(path, "rt") as fh:
        hdr = fh.readline().rstrip("\n").split("\t")
    panel_i = hdr.index("panel_variant_id")
    pos_i = hdr.index("position")
    pval_i = hdr.index("pvalue")
    tb = pysam.TabixFile(path)
    rows = []
    for line in tb.fetch(region=region):
        f = line.split("\t")
        rows.append((f[panel_i], int(f[pos_i]), float(f[pval_i])))
    if not rows:
        return pd.DataFrame(columns=["variant_id", "pos", "logp"])
    df = pd.DataFrame(rows, columns=["variant_id", "pos", "pvalue"])
    df["logp"] = -np.log10(df["pvalue"].clip(lower=1e-300))
    return df[["variant_id", "pos", "logp"]]


def load_eqtl_nominal(tissue_id, gene_id, chrom, win_lo, win_hi):
    path = str(eqtl_nominal_dir / f"{tissue_id}.v11.allpairs.{chrom}.parquet")
    df = pd.read_parquet(path, columns=["variant_id", "pval_nominal"], filters=[("gene_id", "=", gene_id)])
    df["pos"] = df["variant_id"].str.split("_").str[1].astype(int)
    df = df[df["pos"].between(win_lo, win_hi)].copy()
    df["logp"] = -np.log10(df["pval_nominal"].clip(lower=1e-300))
    return df[["variant_id", "pos", "logp"]]


chrom, lead_pos, _, _ = parse_variant_full(cos_lead_variant)
win_lo = lead_pos - locus_win_half
win_hi = lead_pos + locus_win_half
lead_panel_id = coloc_to_panel(cos_lead_variant)

print("Loading LD...", flush=True)
ld_r2, _ = load_ld_r2_for_locus(chrom, cos_lead_variant, win_lo, win_hi)
ld_r2_by_panel = ld_r2.copy()
ld_r2_by_panel.index = ld_r2_by_panel.index.map(
    lambda v: v if str(v).endswith("_b38") else f"{v}_b38"
)

print("Loading GWAS...", flush=True)
gwas_df = load_gwas_nominal(gwas_primary, chrom, win_lo, win_hi)
gwas_df = gwas_df.rename(columns={"logp": "gwas_logp", "variant_id": "panel_id"})

print("Loading eQTLs...", flush=True)
eqtl_tracks = []
eqtl_frames = {}
for i, gid in enumerate(tqdm(gene_ids, desc="eQTL genes")):
    name = gene_display_name(gid)
    col = f"eqtl_logp_{i}"
    df = load_eqtl_nominal(tissue, gid, chrom, win_lo, win_hi).rename(
        columns={"logp": col, "variant_id": "panel_id"}
    )
    eqtl_frames[col] = df[["panel_id", "pos", col]]
    eqtl_tracks.append({
        "col": col,
        "gene_id": gid,
        "label": f"{name} eQTL",
        "name": name,
        "biotype": gene_biotype_key(gid),
    })
    print(f"  {name}: {len(df):,} variants", flush=True)

locus_assoc = gwas_df[["panel_id", "pos", "gwas_logp"]].copy()
for col, df in eqtl_frames.items():
    locus_assoc = locus_assoc.merge(df[["panel_id", col]], on="panel_id", how="outer")
if locus_assoc["pos"].isna().any():
    pos_map = pd.concat(
        [gwas_df[["panel_id", "pos"]]] + [df[["panel_id", "pos"]] for df in eqtl_frames.values()]
    ).drop_duplicates("panel_id")
    locus_assoc = locus_assoc.drop(columns=["pos"]).merge(pos_map, on="panel_id", how="left")

locus_assoc["ld_r2"] = locus_assoc["panel_id"].map(ld_r2_by_panel)
miss = locus_assoc["ld_r2"].isna()
if miss.any():
    alt = locus_assoc.loc[miss, "panel_id"].str.replace("_b38$", "", regex=True).map(ld_r2)
    locus_assoc.loc[miss, "ld_r2"] = alt.values

# Focal CoS VCP from Zenodo long (full_df); p-value tracks already from portal/GWAS nominals.
_vcp = (
    full_df[["variant_id", "vcp"]]
    .drop_duplicates("variant_id")
    .assign(panel_id=lambda d: d["variant_id"].map(coloc_to_panel))
    .set_index("panel_id")["vcp"]
    .astype(float)
)
locus_assoc["vcp"] = locus_assoc["panel_id"].map(_vcp).fillna(0.0)

print(f"Locus assoc table: {len(locus_assoc):,} variants  window={win_lo}-{win_hi}")
print(f"Lead in table: {(locus_assoc.panel_id == lead_panel_id).any()}")
print(f"Focal CoS: {focal_cos_id}; VCP>0 variants: {int((locus_assoc['vcp'] > 0).sum())}")


### Build NPC table across models


In [ ]:
def _parse_shared_gene_gwas(df, model):
    """Rows for CoS that include ≥1 gene expression QTL and ≥1 Astle GWAS trait."""
    rows = []
    ph_col = "phenotype_id" if "phenotype_id" in df.columns else "phenotype_ids"
    for cs, g in df.groupby("cos_id"):
        ph = str(g[ph_col].iloc[0])
        parts = ph.split(";")
        if not any("Astle" in p for p in parts):
            continue
        if not any("expression_" in p for p in parts):
            continue
        npcs = [float(x) for x in str(g.npc_outcomes.iloc[0]).split(";")]
        cos_npc = float(g.cos_npc.iloc[0]) if pd.notna(g.cos_npc.iloc[0]) else np.nan
        for i, p in enumerate(parts):
            if "expression_" not in p:
                continue
            gid = p.split("expression_")[-1]
            rows.append({
                "model": model,
                "cos_id": cs,
                "gene_id": gid,
                "gene_name": gene_display_name(gid),
                "biotype": gene_biotype_key(gid),
                "cos_npc": cos_npc,
                "gene_npc_outcome": npcs[i] if i < len(npcs) else np.nan,
                "shared_with_gwas": True,
            })
    return pd.DataFrame(rows)


def _max_gene_npc_in_df(df, gene_id):
    """Strongest gene npc_outcome for this gene across any CoS/ucos in the run."""
    best = 0.0
    for _, g in df.groupby("cos_id"):
        ph = str(g.phenotype_id.iloc[0])
        if gene_id not in ph:
            continue
        parts = ph.split(";")
        npcs = [float(x) for x in str(g.npc_outcomes.iloc[0]).split(";")]
        for i, p in enumerate(parts):
            if gene_id in p and i < len(npcs):
                best = max(best, npcs[i])
    return best


def _gwas_point_row(df, model, gwas_id=gwas_primary, lead_pos=None, max_dist=50_000):
    """One grey GWAS point: shared CoS npc if present, else univariate GWAS npc near lead."""
    lead_pos = lead_pos if lead_pos is not None else int(cos_lead_variant.split(":")[1])
    ph_col = "phenotype_id" if "phenotype_id" in df.columns else "phenotype_ids"
    best = None
    for cs, g in df.groupby("cos_id"):
        ph = str(g[ph_col].iloc[0])
        parts = ph.split(";")
        if gwas_id not in ph or not any("expression_" in p for p in parts):
            continue
        npcs = [float(x) for x in str(g.npc_outcomes.iloc[0]).split(";")]
        cos_npc = float(g.cos_npc.iloc[0]) if pd.notna(g.cos_npc.iloc[0]) else np.nan
        gwas_npc = np.nan
        for i, p in enumerate(parts):
            if p == gwas_id or p.endswith(gwas_id):
                gwas_npc = npcs[i] if i < len(npcs) else np.nan
                break
        if pd.isna(gwas_npc):
            continue
        cand = {
            "model": model,
            "cos_id": cs,
            "gene_id": gwas_id,
            "gene_name": gwas_short_label(gwas_id),
            "biotype": "gwas",
            "cos_npc": cos_npc,
            "gene_npc_outcome": float(gwas_npc),
            "shared_with_gwas": True,
        }
        if best is None or (pd.notna(cand["cos_npc"]) and (best["cos_npc"] is None or pd.isna(best["cos_npc"]) or cand["cos_npc"] > best["cos_npc"])):
            best = cand
    if best is not None:
        return best

    # Univariate / GWAS-only CoS (Zenodo IDs are tissue_N, not ucos_*).
    gwas_rows = df[
        df[ph_col].astype(str).str.contains(re.escape(gwas_id), regex=True)
        & ~df[ph_col].astype(str).str.contains("expression_", regex=False)
    ].copy()
    npc = 0.0
    cs_id = None
    if not gwas_rows.empty and "npc_outcomes" in gwas_rows.columns:
        vid_col = "lead_variant_id" if "lead_variant_id" in gwas_rows.columns else ("variant_id" if "variant_id" in gwas_rows.columns else None)
        if vid_col is not None:
            gwas_rows = gwas_rows.copy()
            gwas_rows["pos"] = gwas_rows[vid_col].astype(str).str.split(":").str[1].astype(float)
            near = gwas_rows[(gwas_rows["pos"] - lead_pos).abs() <= max_dist]
            use = near if len(near) else gwas_rows
        else:
            use = gwas_rows
        by_cs = use.assign(_npc=use["npc_outcomes"].map(lambda s: min(float(x) for x in str(s).split(";") if str(x).strip()))).groupby("cos_id")["_npc"].max().sort_values(ascending=False)
        cs_id = by_cs.index[0]
        npc = float(by_cs.iloc[0])
    return {
        "model": model,
        "cos_id": cs_id,
        "gene_id": gwas_id,
        "gene_name": gwas_short_label(gwas_id),
        "biotype": "gwas",
        "cos_npc": 0.0,
        "gene_npc_outcome": npc,
        "shared_with_gwas": False,
    }


npc_parts = []

joint_shared = _parse_shared_gene_gwas(full_df_leads, "joint")
if joint_shared.empty:
    # Defensive: attach focal CoS genes even if phenotype parse missed GWAS.
    joint_rows = []
    for gid in gene_ids:
        joint_rows.append({
            "model": "joint",
            "cos_id": focal_cos_id,
            "gene_id": gid,
            "gene_name": gene_display_name(gid),
            "biotype": gene_biotype_key(gid),
            "cos_npc": float(focal_row["cos_npc"]) if pd.notna(focal_row.get("cos_npc")) else 0.0,
            "gene_npc_outcome": _max_gene_npc_in_df(full_df_leads, gid),
            "shared_with_gwas": gwas_primary in str(focal_row.get("phenotype_id", focal_row.get("phenotype_ids", ""))),
        })
    npc_parts.append(pd.DataFrame(joint_rows))
else:
    npc_parts.append(joint_shared)
npc_parts.append(pd.DataFrame([_gwas_point_row(full_df_leads, "joint")]))

pc_shared = _parse_shared_gene_gwas(pc_df, "pc_only")
if pc_shared.empty:
    pc_rows = []
    for gid in ["ENSG00000184221.13", "ENSG00000205927.5"]:
        pc_rows.append({
            "model": "pc_only",
            "cos_id": None,
            "gene_id": gid,
            "gene_name": gene_display_name(gid),
            "biotype": gene_biotype_key(gid),
            "cos_npc": 0.0,
            "gene_npc_outcome": _max_gene_npc_in_df(pc_df, gid),
            "shared_with_gwas": False,
        })
    npc_parts.append(pd.DataFrame(pc_rows))
else:
    npc_parts.append(pc_shared)
npc_parts.append(pd.DataFrame([_gwas_point_row(pc_df, "pc_only")]))

v39_shared = _parse_shared_gene_gwas(v39_df, "v39_only")
if v39_shared.empty:
    v39_rows = []
    for gid in gene_ids:
        # only_v39 drops novel_lncRNA; skip genes absent from this run
        if gene_biotype_key(gid) == "novel_lncRNA":
            continue
        if not v39_df["phenotype_id"].astype(str).str.contains(re.escape(gid), regex=True).any():
            continue
        v39_rows.append({
            "model": "v39_only",
            "cos_id": None,
            "gene_id": gid,
            "gene_name": gene_display_name(gid),
            "biotype": gene_biotype_key(gid),
            "cos_npc": 0.0,
            "gene_npc_outcome": _max_gene_npc_in_df(v39_df, gid),
            "shared_with_gwas": False,
        })
    npc_parts.append(pd.DataFrame(v39_rows))
else:
    npc_parts.append(v39_shared)
npc_parts.append(pd.DataFrame([_gwas_point_row(v39_df, "v39_only")]))

# Single-phenotype discovery leads (Zenodo) instead of per-job ColocBoost .txt files.
ind_rows = []
shared_all = _parse_shared_gene_gwas(sg_df, "individual")
if not shared_all.empty:
    ind_rows.extend(shared_all.to_dict("records"))
# Include each focal gene even if it is not recovered with GWAS in this mode.
have = {str(r.get("gene_id", "")).split(".")[0] for r in ind_rows}
for gid in gene_ids:
    core = gid.split(".")[0]
    if any(core in str(h) for h in have):
        continue
    ind_rows.append({
        "model": "individual",
        "cos_id": None,
        "gene_id": gid,
        "gene_name": gene_display_name(gid),
        "biotype": gene_biotype_key(gid),
        "cos_npc": 0.0,
        "gene_npc_outcome": _max_gene_npc_in_df(sg_df, gid),
        "shared_with_gwas": False,
    })
npc_parts.append(pd.DataFrame(ind_rows))
if len(sg_df):
    npc_parts.append(pd.DataFrame([_gwas_point_row(sg_df, "individual")]))

npc_df = pd.concat(npc_parts, ignore_index=True)
npc_df = npc_df.drop_duplicates(["model", "gene_id"], keep="first")


## Make figure


In [ ]:
def _load_locus_exons(genes_df, chrom, win_lo, win_hi):
    """Representative-transcript exons for genes in window."""
    if genes_df.empty:
        return pd.DataFrame(columns=["gene_id", "transcript_id", "Start", "End"])
    gids = set(genes_df["gene_id"])
    stables = set(genes_df["stable"])
    exon_df = ann[
        (ann["Feature"] == "exon")
        & (ann["Chromosome"] == chrom)
        & (ann["Start"] <= win_hi)
        & (ann["End"] >= win_lo)
    ].copy()
    exon_df = exon_df[
        exon_df["gene_id"].isin(gids)
        | exon_df["gene_id"].astype(str).str.split(".").str[0].isin(stables)
    ]
    if exon_df.empty:
        return exon_df
    exon_df = exon_df.assign(exon_len=lambda d: d["End"] - d["Start"])
    rep_tx = (
        exon_df.groupby(["gene_id", "transcript_id"], as_index=False)["exon_len"]
        .sum()
        .sort_values("exon_len", ascending=False)
        .drop_duplicates("gene_id")
    )
    return exon_df.merge(rep_tx[["gene_id", "transcript_id"]], on=["gene_id", "transcript_id"])


def _ld_colors(vals):
    return ["#1a1a1a" if pd.isna(r2) else ld_cmap(ld_norm(float(r2))) for r2 in vals]


def _npc_panel_units(npc_df):
    """Display order: CoS genes then GWAS."""
    name_order = [gene_display_name(g) for g in gene_ids] + [gwas_short_label(gwas_primary)]
    present = set(npc_df["gene_name"])
    names = [n for n in name_order if n in present]
    for n in npc_df["gene_name"].unique():
        if n not in names:
            names.append(n)
    return names


# Layout and draw panels a-e
tracks_n = 1 + len(eqtl_tracks)
n_cmp = len(eqtl_tracks)
cmp_nrows = 2 if n_cmp > 2 else 1
cmp_ncols = int(np.ceil(n_cmp / cmp_nrows))
n_left = tracks_n + 3

fig = plt.figure(figsize=(fig_width, fig_height), dpi=fig_dpi)
outer = fig.add_gridspec(
    1, 2,
    width_ratios=[1.30, 1.25],
    wspace=0.28,
    left=0.10, right=0.98, top=0.96, bottom=0.06,
)
left = outer[0].subgridspec(
    n_left, 1,
    height_ratios=[0.55] * tracks_n + [0.28, 0.70, 1.05],
    hspace=0.10,
)
right = outer[1].subgridspec(
    3, 1,
    height_ratios=[0.75, 1.45, 1.05],
    hspace=0.42,
)

locus_axes = [fig.add_subplot(left[0])]
for i in range(1, tracks_n + 2):
    locus_axes.append(fig.add_subplot(left[i], sharex=locus_axes[0]))
axz = fig.add_subplot(left[tracks_n + 2])
locus_axes.append(axz)

ax_expr = fig.add_subplot(right[0])
cmp_grid = right[1].subgridspec(cmp_nrows, cmp_ncols, wspace=0.38, hspace=0.85)

# Shared axis limits for the GWAS vs eQTL comparison scatters
xmax = ymax = 1.0
for tr in eqtl_tracks:
    sub = locus_assoc[
        locus_assoc["gwas_logp"].notna() & locus_assoc[tr["col"]].notna()
    ]
    if len(sub):
        xmax = max(xmax, float(np.nanmax(sub[tr["col"]].values)))
        ymax = max(ymax, float(np.nanmax(sub["gwas_logp"].values)))
xlim = (0, xmax * 1.08)
ylim = (0, ymax * 1.08)

# Panel a: locus association tracks, VCP, gene tracks, zoom
tracks = [{'col': 'gwas_logp', 'label': gwas_short_label(gwas_primary)}] + [{'col': t['col'], 'label': t['label']} for t in eqtl_tracks]
n_assoc = len(tracks)
assoc_axes = locus_axes[:n_assoc]
axv = locus_axes[n_assoc]
axg = locus_axes[n_assoc + 1]
axz = locus_axes[n_assoc + 2]
coset_stables = {g.split('.')[0] for g in gene_ids}
genes_in_win = gene_ann[(gene_ann['Chromosome'] == chrom) & (gene_ann['Start'] <= win_hi) & (gene_ann['End'] >= win_lo)].drop_duplicates('stable')
genes_in_win = pd.concat([genes_in_win[~genes_in_win['stable'].isin(coset_stables)], genes_in_win[genes_in_win['stable'].isin(coset_stables)]], ignore_index=True)
exons_in_win = _load_locus_exons(genes_in_win, chrom, win_lo, win_hi)
sub = locus_assoc[locus_assoc['pos'].between(win_lo, win_hi)].copy()

# Shared y-limit across association tracks
assoc_ymax = 1.0
for tr in tracks:
    y = sub[tr['col']]
    if y.notna().any():
        assoc_ymax = max(assoc_ymax, float(np.nanmax(y.values)))
assoc_ylim = (0, assoc_ymax * 1.12)

# GWAS and eQTL association tracks
for ax, tr in zip(assoc_axes, tracks):
    y = sub[tr['col']]
    mask = y.notna()
    ax.scatter(sub.loc[mask, 'pos'], y[mask], s=base_marker_s, c=_ld_colors(sub.loc[mask, 'ld_r2']), linewidths=0, alpha=0.9, zorder=3)
    lead_row = sub[sub['panel_id'] == lead_panel_id]
    if len(lead_row) and pd.notna(lead_row.iloc[0][tr['col']]):
        ax.scatter([lead_row.iloc[0]['pos']], [lead_row.iloc[0][tr['col']]], c='#734675', s=10, marker='D', edgecolors='none', zorder=10)
    _lab = tr['label']
    ylab = f'{_lab[:-5]}\neQTL\n$-\\log_{{10}}(p)$' if _lab.endswith(' eQTL') else f'{_lab}\n$-\\log_{{10}}(p)$'
    ax.set_ylabel(ylab, rotation=0, ha='right', va='center', labelpad=4, fontsize=4)
    ax.set_xlim(win_lo, win_hi)
    ax.set_ylim(*assoc_ylim)
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    sns.despine(ax=ax, bottom=False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_linewidth(0.6)
    ax.tick_params(axis='x', bottom=False, labelbottom=False, length=0)
    ax.tick_params(axis='y', labelsize=4, pad=1)
ax0 = assoc_axes[0]
ax0.xaxis.set_major_locator(plt.MaxNLocator(nbins=4))
ax0.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax0.xaxis.tick_top()
ax0.tick_params(axis='x', which='major', top=True, labeltop=True, bottom=False, labelbottom=False, direction='out', length=1.8, width=0.45, pad=1, colors='0.15', labelsize=4)
ax0.spines['top'].set_visible(True)
ax0.spines['top'].set_linewidth(0.6)
ax0.text(0.0, 1.02, chrom, transform=ax0.transAxes, ha='left', va='bottom', fontsize=5, color='0.2', clip_on=False)

# VCP track
cs_sub = sub[sub['vcp'] > 0]
if len(cs_sub):
    axv.scatter(cs_sub['pos'], cs_sub['vcp'], s=base_marker_s, c=_ld_colors(cs_sub['ld_r2']), linewidths=0, alpha=0.9, zorder=3)
    lead_row = cs_sub[cs_sub['panel_id'] == lead_panel_id]
    if len(lead_row):
        axv.scatter([lead_row.iloc[0]['pos']], [float(lead_row.iloc[0]['vcp'])], c='#734675', s=10, marker='D', edgecolors='none', zorder=10)
axv.set_ylabel('VCP', rotation=0, ha='right', va='center', labelpad=4, fontsize=4)
axv.set_xlim(win_lo, win_hi)
axv.set_ylim(0, 1.05)
axv.set_yticks([0, 1.0])
sns.despine(ax=axv, bottom=False)
axv.spines['bottom'].set_visible(True)
axv.spines['bottom'].set_linewidth(0.6)
axv.tick_params(axis='x', bottom=False, labelbottom=False, length=0)
axv.tick_params(axis='y', labelsize=4, pad=1)

# Zoom window bounds around the CoS genes
pad = 5000
rows = gene_ann[gene_ann['stable'].isin(coset_stables)]
coords = [int(lead_pos)]
for _, r in rows.iterrows():
    s, e = (int(r['Start']), int(r['End']))
    if e - s > 80000:
        ex = ann[(ann['Feature'] == 'exon') & (ann['gene_id'].astype(str).str.split('.').str[0] == r['stable']) & (ann['Chromosome'] == chrom)]
        for a, b in zip(ex['Start'], ex['End']):
            a, b = (int(a), int(b))
            if b >= lead_pos - 60000 and a <= lead_pos + 20000:
                coords.extend([a, b])
    else:
        coords.extend([s, e])
if len(coords) <= 1:
    z_lo, z_hi = (lead_pos - zoom_win_half, lead_pos + zoom_win_half)
else:
    z_lo, z_hi = (min(coords) - pad, max(coords) + pad)
z_lo = max(z_lo, win_lo)
z_hi = min(z_hi, win_hi)

# Overview gene track
ax = axg
genes_df = genes_in_win
exons_df = exons_in_win
label_left_of = z_lo
y_plus, y_minus = (0.68, 0.28)
exon_h = 0.12
for y in (y_plus, y_minus):
    ax.axhline(y, color='0.85', lw=0.4, zorder=0)
strand_trans = blended_transform_factory(ax.transAxes, ax.transData)
ax.text(0.995, y_plus + 0.05, '+', transform=strand_trans, ha='right', va='bottom', fontsize=5, color='0.35', clip_on=False)
ax.text(0.995, y_minus + 0.05, '−', transform=strand_trans, ha='right', va='bottom', fontsize=5, color='0.35', clip_on=False)
span = max(win_hi - win_lo, 1)
min_exon_w = 0.004 * span

def draw_one(gene_row, y, label_y, label_va='bottom', label_x=None, label_ha='center', *, draw_label=True):
    gid = gene_row['gene_id']
    stable = str(gid).split('.')[0]
    in_coset = stable in coset_stables
    color = gene_type_palette.get(gene_biotype_key(gid), '#666666')
    lw = 0.45
    fs = 3.6 if in_coset else 3.1
    z_body = 3 if in_coset else 1
    g0 = max(int(gene_row['Start']), win_lo)
    g1 = min(int(gene_row['End']), win_hi)
    if g1 <= g0:
        return
    ax.plot([g0, g1], [y, y], color=color, lw=lw, solid_capstyle='butt', zorder=z_body)
    gene_exons = exons_df[exons_df['gene_id'] == gid].sort_values('Start')
    if gene_exons.empty:
        ax.add_patch(plt.Rectangle((g0, y - exon_h / 2), max(g1 - g0, min_exon_w), exon_h, facecolor=color, edgecolor='none', zorder=z_body + 1, linewidth=0))
    else:
        for _, ex in gene_exons.iterrows():
            x0 = max(int(ex['Start']), win_lo)
            x1 = min(int(ex['End']), win_hi)
            if x1 <= x0:
                continue
            w = max(x1 - x0, min_exon_w)
            ax.add_patch(plt.Rectangle((x0, y - exon_h / 2), w, exon_h, facecolor=color, edgecolor='none', zorder=z_body + 1, linewidth=0))
    if not draw_label:
        return
    label = gene_display_name(gid) if in_coset else gene_row['gene_name']
    lx = 0.5 * (g0 + g1) if label_x is None else label_x
    lha = label_ha
    if in_coset:
        pad = 0.012 * span
        lx = min(lx, float(label_left_of) - pad)
        lha = 'right'
    ax.text(lx, label_y, label, ha=lha, va=label_va, fontsize=fs, color=color, style='italic', clip_on=False, zorder=z_body + 3)

ordered = pd.concat([genes_df[genes_df['stable'].isin(coset_stables)].sort_values('Start'), genes_df[~genes_df['stable'].isin(coset_stables)].sort_values('Start')], ignore_index=True)
placed = {'+': [], '-': []}
label_pad = 0.1
slot_dy = 0.13
for _, grow in ordered.iterrows():
    strand = grow['Strand'] if grow['Strand'] in ('+', '-') else '+'
    y = y_plus if strand == '+' else y_minus
    g0 = max(int(grow['Start']), win_lo)
    g1 = min(int(grow['End']), win_hi)
    if g1 <= g0:
        continue
    stable = str(grow['gene_id']).split('.')[0]
    in_coset = stable in coset_stables
    label = gene_display_name(grow['gene_id']) if in_coset else grow['gene_name']
    gene_exons = exons_in_win[exons_in_win['gene_id'] == grow['gene_id']]
    if not gene_exons.empty:
        eg0 = max(int(gene_exons['Start'].min()), win_lo)
        eg1 = min(int(gene_exons['End'].max()), win_hi)
        mid = 0.5 * (eg0 + eg1)
    else:
        mid = g0 + 0.35 * (g1 - g0) if strand == '+' else 0.5 * (g0 + g1)
    half_w = max(0.035 * span, 0.009 * span * max(len(str(label)), 4))
    x0, x1 = (mid - half_w, mid + half_w)
    slot = 0
    max_slot = 4 if in_coset else 2
    while slot <= max_slot:
        hit = any((slot == ps and (not (x1 < px0 or x0 > px1)) for px0, px1, ps in placed[strand]))
        if not hit:
            break
        slot += 1
    if slot > max_slot and (not in_coset):
        draw_one(grow, y, label_y=y, label_va='center', draw_label=False)
        continue
    slot = min(slot, max_slot)
    placed[strand].append((x0, x1, slot))
    if strand == '+':
        label_y = y + label_pad + slot * slot_dy
        label_va = 'bottom'
    else:
        label_y = y - label_pad - slot * slot_dy
        label_va = 'top'
    draw_one(grow, y, label_y=label_y, label_va=label_va, label_x=mid, label_ha='center')
ax.set_ylim(-0.15, 1.25)
axg.set_xlim(win_lo, win_hi)
axg.set_yticks([])
axg.set_ylabel('')
sns.despine(ax=axg, left=True, bottom=True)
axg.tick_params(axis='x', bottom=False, labelbottom=False, length=0)

# Zoomed CoS gene track
ax = axz
zoom_lo = z_lo
zoom_hi = z_hi
genes = gene_ann[gene_ann['stable'].isin(coset_stables)].drop_duplicates('stable').sort_values('Start').reset_index(drop=True)
exons_df = _load_locus_exons(genes, chrom, zoom_lo, zoom_hi)
y_cos, y_plus, y_minus = (0.92, 0.58, 0.22)
exon_h = 0.08
span = max(zoom_hi - zoom_lo, 1)
min_exon_w = 0.006 * span
lw = 0.45
ax.axhline(y_cos, color='0.85', lw=0.4, zorder=0)
for y in (y_plus, y_minus):
    ax.axhline(y, color='0.85', lw=0.4, zorder=0)
strand_trans = blended_transform_factory(ax.transAxes, ax.transData)
ax.text(0.995, y_plus + 0.05, '+', transform=strand_trans, ha='right', va='bottom', fontsize=5, color='0.35', clip_on=False)
ax.text(0.995, y_minus + 0.05, '−', transform=strand_trans, ha='right', va='bottom', fontsize=5, color='0.35', clip_on=False)
placed = {'+': [], '-': []}
for _, grow in genes.iterrows():
    gid = grow['gene_id']
    color = gene_type_palette.get(gene_biotype_key(gid), '#666666')
    strand = grow['Strand'] if grow['Strand'] in ('+', '-') else '+'
    y = y_plus if strand == '+' else y_minus
    g0 = max(int(grow['Start']), zoom_lo)
    g1 = min(int(grow['End']), zoom_hi)
    if g1 <= g0:
        continue
    gene_exons = exons_df[exons_df['gene_id'] == gid].sort_values('Start')
    exon_ivs = []
    for _, ex in gene_exons.iterrows():
        x0 = max(int(ex['Start']), zoom_lo)
        x1 = min(int(ex['End']), zoom_hi)
        if x1 > x0:
            exon_ivs.append((x0, x1))
    if exon_ivs:
        ax.plot([exon_ivs[0][0], exon_ivs[-1][1]], [y, y], color=color, lw=lw, solid_capstyle='butt', zorder=2)
        for x0, x1 in exon_ivs:
            ax.add_patch(plt.Rectangle((x0, y - exon_h / 2), max(x1 - x0, min_exon_w), exon_h, facecolor=color, edgecolor='none', zorder=3, linewidth=0))
    else:
        ax.plot([g0, g1], [y, y], color=color, lw=lw, solid_capstyle='butt', zorder=2)
        ax.add_patch(plt.Rectangle((g0, y - exon_h / 2), max(g1 - g0, min_exon_w), exon_h, facecolor=color, edgecolor='none', zorder=3, linewidth=0))
    name = gene_display_name(gid)
    # Anchor label to visible exons (gene Start/End can be much wider than drawn exons).
    if exon_ivs:
        mid = 0.5 * (exon_ivs[0][0] + exon_ivs[-1][1])
    else:
        mid = 0.5 * (g0 + g1)
    half_w = max(0.04 * span, 0.01 * span * max(len(str(name)), 4))
    slot = 0
    for px0, px1, ps in placed[strand]:
        if not (mid + half_w < px0 or mid - half_w > px1):
            slot = max(slot, ps + 1)
    placed[strand].append((mid - half_w, mid + half_w, slot))
    if strand == '+':
        label_y = y + 0.1 + slot * 0.11
        label_va = 'bottom'
    else:
        label_y = y - 0.1 - slot * 0.11
        label_va = 'top'
    ax.text(mid, label_y, name, ha='center', va=label_va, fontsize=3.4, color=color, style='italic', clip_on=False, zorder=6)
vcp_sub = locus_assoc[locus_assoc['pos'].between(zoom_lo, zoom_hi) & (locus_assoc['vcp'] > 0)]
if len(vcp_sub):
    ax.scatter(vcp_sub['pos'], np.full(len(vcp_sub), y_cos), s=base_marker_s * 1.3, c=_ld_colors(vcp_sub['ld_r2']), linewidths=0, alpha=0.9, zorder=4)
    lead_row = vcp_sub[vcp_sub['panel_id'] == lead_panel_id]
    if len(lead_row):
        lead_x = float(lead_row.iloc[0]['pos'])
        ax.scatter([lead_x], [y_cos], c='#734675', s=10, marker='D', edgecolors='none', zorder=10)
        ax.text(lead_x, y_cos + 0.045, 'CoS lead', ha='center', va='bottom', fontsize=3.2, color='0.1', clip_on=False, zorder=6)
ax.set_xlim(zoom_lo, zoom_hi)
ax.set_ylim(0.0, 1.08)
ax.set_yticks([])
ax.set_ylabel('')
ax.tick_params(axis='x', bottom=False, labelbottom=False, length=0)
sns.despine(ax=ax, left=True, bottom=True)
bar = 5000 if span >= 40000 else 2000 if span >= 15000 else 1000
bx0 = zoom_hi - bar - 0.04 * span
ax.plot([bx0, bx0 + bar], [0.06, 0.06], color='0.2', lw=0.9, clip_on=False)
ax.text(bx0 + 0.5 * bar, 0.03, f'{bar // 1000} kb', ha='center', va='top', fontsize=4, color='0.25')

# Connector shading between overview and zoom tracks
ax_overview = axg
ax_zoom = axz
span_axes = [axg]
fill_rgb = mcolors.to_rgb('0.72')
line_rgb = mcolors.to_rgb('0.55')
span_alpha = 0.16
line_alpha = 0.85
lw = 0.55
n_fade = 40

def _lerp(a, b, t):
    return (a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t)

def _to_fig(ax, x, y):
    return fig.transFigure.inverted().transform(ax.transData.transform((x, y)))
for ax in span_axes:
    ax.axvspan(zoom_lo, zoom_hi, facecolor=fill_rgb, edgecolor='none', alpha=span_alpha, zorder=0, lw=0)
    y_lo, y_hi = ax.get_ylim()
    for x in (zoom_lo, zoom_hi):
        ax.plot([x, x], [y_lo, y_hi], color=line_rgb, lw=lw, ls='-', solid_capstyle='butt', zorder=0, alpha=line_alpha)
ov_y0 = ax_overview.get_ylim()[0]
z_y0, z_y1 = ax_zoom.get_ylim()
ov_bl, ov_br = (_to_fig(ax_overview, zoom_lo, ov_y0), _to_fig(ax_overview, zoom_hi, ov_y0))
zm_tl, zm_tr = (_to_fig(ax_zoom, zoom_lo, z_y1), _to_fig(ax_zoom, zoom_hi, z_y1))
for i in range(n_fade):
    t0, t1 = (i / n_fade, (i + 1) / n_fade)
    fig.add_artist(Polygon([_lerp(ov_bl, zm_tl, t0), _lerp(ov_br, zm_tr, t0), _lerp(ov_br, zm_tr, t1), _lerp(ov_bl, zm_tl, t1)], transform=fig.transFigure, facecolor=fill_rgb, edgecolor='none', alpha=span_alpha, zorder=0, clip_on=False))
for p0, p1 in ((ov_bl, zm_tl), (ov_br, zm_tr)):
    segs = [[_lerp(p0, p1, i / n_fade), _lerp(p0, p1, (i + 1) / n_fade)] for i in range(n_fade)]
    fig.add_artist(LineCollection(segs, colors=[(*line_rgb, line_alpha)] * n_fade, linewidths=lw, transform=fig.transFigure, zorder=1, clip_on=False, capstyle='butt'))
for i in range(n_fade):
    t0, t1 = (i / n_fade, (i + 1) / n_fade)
    tm = 0.5 * (t0 + t1)
    y_a = z_y1 + (z_y0 - z_y1) * t0
    y_b = z_y1 + (z_y0 - z_y1) * t1
    ax_zoom.axhspan(min(y_a, y_b), max(y_a, y_b), facecolor=fill_rgb, edgecolor='none', alpha=span_alpha * (1.0 - tm), zorder=0, lw=0)
for x in (zoom_lo, zoom_hi):
    segs, cols = ([], [])
    for i in range(n_fade):
        t0, t1 = (i / n_fade, (i + 1) / n_fade)
        tm = 0.5 * (t0 + t1)
        y_a = z_y1 + (z_y0 - z_y1) * t0
        y_b = z_y1 + (z_y0 - z_y1) * t1
        segs.append([(x, y_a), (x, y_b)])
        cols.append((*line_rgb, line_alpha * (1.0 - tm)))
    ax_zoom.add_collection(LineCollection(segs, colors=cols, linewidths=lw, zorder=0, capstyle='butt'))
ax0.xaxis.tick_top()
ax0.tick_params(axis='x', which='major', top=True, labeltop=True, bottom=False, labelbottom=False, direction='out', length=1.8, width=0.45, pad=1, colors='0.15', labelsize=4)

# LD legend on top association track
ax0.legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=3, label=lab) for c, lab in zip(ld_colors, ['0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1'])] + [Line2D([0], [0], marker='D', color='w', markerfacecolor='#734675', markeredgecolor='none', markersize=3, label='CoS lead')], title='LD $r^2$', loc='upper right', frameon=True, fancybox=False, edgecolor='0.7', framealpha=0.95, fontsize=4, title_fontsize=5, handletextpad=0.3, labelspacing=0.2, borderpad=0.35)

# Panel c: expression heatmap
ax = ax_expr
mat = expr_log.to_numpy(dtype=float)
genes = list(expr_log.index)
tissues = list(expr_log.columns)
im = ax.imshow(mat, aspect='auto', interpolation='nearest', cmap='Blues', vmin=0, vmax=max(1.0, float(np.nanmax(mat))))
ax.set_yticks(np.arange(len(genes)))
ylabels = [gene_display_name(g) for g in genes]
ax.set_yticklabels(ylabels, fontsize=3.8)
for lab in ax.get_yticklabels():
    lab.set_color('#222222')
    lab.set_fontstyle('italic')
ax.set_xticks(np.arange(len(tissues)))
ax.set_xticklabels([tissue_abbr.get(t, t[:4]) for t in tissues], fontsize=2.8, rotation=90, ha='center', va='top')
for tick, t in zip(ax.get_xticklabels(), tissues):
    tick.set_color(tissue_color.get(t, '#333333'))
wb = 'Whole Blood'
if wb in tissues:
    j = tissues.index(wb)
    ax.add_patch(Rectangle((j - 0.5, -0.5), 1.0, len(genes), fill=False, edgecolor='0.1', linewidth=1.0, zorder=5, clip_on=False))
ax.set_ylabel('')
ax.set_xlabel('')
ax.tick_params(length=1.2, pad=1)
cax = ax.inset_axes([1.01, 0.1, 0.02, 0.8])
cb = plt.colorbar(im, cax=cax)
cb.set_label('$\\log_{10}(\\mathrm{TPM}+1)$', fontsize=4.5)
cb.ax.tick_params(labelsize=3.5, length=1.2, pad=1)

# Panel d: GWAS vs eQTL comparison scatters with CoS_NPC annotations
gwas_lab = gwas_short_label(gwas_primary)
cmp_axes = []
for i, tr in enumerate(eqtl_tracks):
    r, c = divmod(i, cmp_ncols)
    ax = fig.add_subplot(cmp_grid[r, c])
    show_ylab = c == 0
    show_xlab = r == cmp_nrows - 1
    sub = locus_assoc[locus_assoc['gwas_logp'].notna() & locus_assoc[tr['col']].notna()].copy()
    is_lead = sub['panel_id'].to_numpy() == lead_panel_id
    sub_bg = sub.loc[~is_lead]
    sub_ld = sub.loc[is_lead]
    if len(sub_bg):
        ax.scatter(sub_bg[tr['col']], sub_bg['gwas_logp'], s=base_marker_s, c=_ld_colors(sub_bg['ld_r2']), linewidths=0, alpha=0.85, zorder=5)
    if len(sub_ld):
        lead_r = sub_ld.iloc[0]
        ax.scatter([lead_r[tr['col']]], [lead_r['gwas_logp']], c='#734675', s=10, marker='D', edgecolors='none', zorder=10)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    if show_xlab:
        ax.set_xlabel('eQTL $-\\log_{10}(P)$', fontsize=4.5)
    else:
        ax.set_xlabel('')
    if show_ylab:
        ax.set_ylabel(f'{gwas_lab} $-\\log_{{10}}(P)$', fontsize=4.5)
    else:
        ax.set_ylabel('')
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3))
    ax.tick_params(labelsize=4.5)
    # CoS_NPC summary text per model
    gid = tr.get('gene_id')
    if gid:
        bt = gene_biotype_key(gid)
        is_pc = bt == 'protein_coding'
        stable = str(gid).split('.')[0]
        model_labs = [('individual', 'single phenotype'), ('pc_only', 'PC-only'), ('v39_only', 'no-novel'), ('joint', 'all genes')]
        lines = ['$CoS_{NPC}$']
        for model, lab in model_labs:
            if model == 'pc_only' and (not is_pc):
                lines.append(f'{lab} —')
                continue
            if model == 'v39_only' and bt == 'novel_lncRNA':
                lines.append(f'{lab} —')
                continue
            if npc_df is None or getattr(npc_df, 'empty', True):
                v = 0.0
            else:
                npc_sub = npc_df[(npc_df['model'] == model) & npc_df['gene_id'].astype(str).str.split('.').str[0].eq(stable) & (npc_df['biotype'] != 'gwas')]
                if npc_sub.empty:
                    v = 0.0
                else:
                    vals = pd.to_numeric(npc_sub['cos_npc'], errors='coerce').dropna()
                    v = float(vals.max()) if len(vals) else 0.0
            lines.append(f'{lab} {v:.2f}')
        ax.text(0.97, 0.03, '\n'.join(lines), transform=ax.transAxes, ha='right', va='bottom', fontsize=3.2, color='0.15', linespacing=1.15, bbox=dict(boxstyle='round,pad=0.18', facecolor='white', edgecolor='0.85', lw=0.35), zorder=8)
    sns.despine(ax=ax)
    ax.set_title(tr["name"], fontsize=5.5, pad=1)
    cmp_axes.append(ax)

# Panel e: CoS_NPC vs NPC_outcome by model
model_order = ["individual", "pc_only", "v39_only", "joint"]
model_titles = {
    "individual": "single phenotype",
    "pc_only": "p.c. only",
    "v39_only": "no-novel",
    "joint": "all genes",
}
npc_plot = npc_df[npc_df["model"].isin(model_order)].copy()
npc_plot["cos_npc"] = pd.to_numeric(npc_plot["cos_npc"], errors="coerce")
npc_plot["gene_npc_outcome"] = pd.to_numeric(npc_plot["gene_npc_outcome"], errors="coerce")
entity_names = _npc_panel_units(npc_plot)
gwas_name = gwas_short_label(gwas_primary)
name_bt = npc_plot.drop_duplicates("gene_name").set_index("gene_name")["biotype"].to_dict()
entity_styles = {}
pc_i = 0
marker_cycle = {
    "protein_coding": ["o", "s"],
    "existing_lncRNA": "^",
    "novel_lncRNA": "D",
    "lncRNA": "D",
    "gwas": "X",
}
for name in entity_names:
    bt = name_bt.get(name)
    if bt is None:
        bt = "gwas" if name == gwas_name else "lncRNA"
    if bt == "protein_coding":
        color = pc_blue_shades[pc_i % len(pc_blue_shades)]
        marker = marker_cycle["protein_coding"][pc_i % 2]
        pc_i += 1
    elif bt == "existing_lncRNA":
        color = gene_type_palette["existing_lncRNA"]
        marker = marker_cycle["existing_lncRNA"]
    elif bt in ("novel_lncRNA", "lncRNA"):
        color = gene_type_palette["novel_lncRNA"]
        marker = marker_cycle["novel_lncRNA"]
    else:
        color = gene_type_palette.get("gwas", "#222222")
        marker = marker_cycle["gwas"]
    entity_styles[name] = {"color": color, "marker": marker}

inner = right[2].subgridspec(1, 4, wspace=0.32)
npc_axes = []
for i, model in enumerate(model_order):
    ax = fig.add_subplot(
        inner[0, i],
        sharex=npc_axes[0] if npc_axes else None,
        sharey=npc_axes[0] if npc_axes else None,
    )
    model_rows = npc_plot[npc_plot["model"] == model]
    ax.fill_between(
        [npc_outcome_min, 1.0], cos_npc_thresh, 1.0, color="#e8f0e8", lw=0, zorder=0
    )
    ax.axhline(cos_npc_thresh, color="#333333", ls="--", lw=0.55, alpha=0.75, zorder=1)
    ax.axvline(npc_outcome_min, color="#333333", ls="--", lw=0.55, alpha=0.75, zorder=1)
    for name in _npc_panel_units(model_rows):
        rows = model_rows[model_rows["gene_name"] == name]
        if rows.empty:
            continue
        row = rows.iloc[0]
        x = float(row["gene_npc_outcome"])
        y = float(row["cos_npc"])
        if not np.isfinite(x) or not np.isfinite(y):
            continue
        style = entity_styles.get(name, {"color": "#888888", "marker": "o"})
        ax.scatter(
            [x], [y], s=22, marker=style["marker"], c=[style["color"]],
            edgecolors="white", linewidths=0.35, zorder=4, clip_on=False,
        )
    pad = 0.06
    ax.set_xlim(-pad, 1 + pad)
    ax.set_ylim(-pad, 1 + pad)
    ax.set_box_aspect(1)
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])
    ax.tick_params(labelsize=4.2)
    ax.set_title(model_titles[model], fontsize=5.5, pad=1.5)
    ax.set_xlabel("$NPC_{outcome}$", fontsize=4.5)
    ax.set_ylabel("$CoS_{NPC}$" if i == 0 else "", fontsize=4.5)
    sns.despine(ax=ax)
    if i > 0:
        ax.tick_params(labelleft=False)
    npc_axes.append(ax)

handles = []
for name in entity_names:
    style = entity_styles[name]
    handles.append(
        Line2D(
            [0], [0], marker=style["marker"], color="w",
            markerfacecolor=style["color"], markeredgecolor="white",
            markeredgewidth=0.3, markersize=4.5, label=name,
        )
    )
leg = npc_axes[1].legend(
    handles=handles, frameon=False, fontsize=3.5, loc="lower center",
    bbox_to_anchor=(1.05, 1.28), ncol=5, labelspacing=0.15,
    handletextpad=0.2, columnspacing=0.55,
)
for text in leg.get_texts():
    if text.get_text() != gwas_name:
        text.set_fontstyle("italic")

# Panel letters
_inv = fig.transFigure.inverted()
# c/d share ax_expr's left so nested cmp subplots don't shift d
_cd_x = _inv.transform(ax_expr.transAxes.transform((-0.08, 0)))[0]
_letter_specs = [
    (locus_axes[0], "a", -0.08, 1.12, None),
    (axz, "b", -0.10, 1.15, None),
    (ax_expr, "c", -0.08, 1.12, _cd_x),
    (cmp_axes[0], "d", -0.08, 1.15, _cd_x),
    (npc_axes[0], "e", -0.22, 1.02, None),
]
for ax, label, x_off, y_ax, x_fig in _letter_specs:
    _x = x_fig if x_fig is not None else _inv.transform(ax.transAxes.transform((x_off, 0)))[0]
    _y = _inv.transform(ax.transAxes.transform((0, y_ax)))[1]
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()

## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / "supp_figure_18_joint_coloc_example.pdf", bbox_inches="tight")
fig.savefig(fig_dir / "supp_figure_18_joint_coloc_example.png", bbox_inches="tight")
print(f"Wrote {fig_dir / 'supp_figure_18_joint_coloc_example.pdf'}")
print(f"Wrote {fig_dir / 'supp_figure_18_joint_coloc_example.png'}")

## Print stats


In [ ]:
# CoS lead variant and genes in the joint CoS
print(f"Lead variant: {cos_lead_variant}")
print(f"Genes in joint CoS: {', '.join(gene_display_name(g) for g in gene_ids)}")

joint_genes = npc_df[(npc_df["model"] == "joint") & (npc_df["biotype"] != "gwas")]
if joint_genes.empty:
    print("Joint CoSnpc: (no shared gene×GWAS CoS)")
else:
    print(f"Joint CoSnpc: {float(pd.to_numeric(joint_genes['cos_npc'], errors='coerce').max()):.2f}")

# Models in which the GWAS signal shares a CoS with the QTL genes
for model, label in [
    ("joint", "joint, all genes"),
    ("pc_only", "joint, protein-coding only"),
    ("individual", "single-phenotype"),
]:
    shared = bool(npc_df.loc[npc_df["model"] == model, "shared_with_gwas"].any())
    print(f"Colocalization recovered, {label}: {shared}")
